In [5]:
import pandas as pd
from sqlalchemy import create_engine
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
import joblib

# =====================
# 1. Load Data from DB
# =====================
db_user = "farmlinkdb3_user"
db_password = "jz3I75kAtGI60UmmBN8Li9816H9sks4v"
db_host = "dpg-d2fdscruibrs739r9o0g-a.oregon-postgres.render.com"
db_port = "5432"
db_name = "farmlinkdb3"

connection_string = f"postgresql+psycopg2://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}"
engine = create_engine(connection_string)

# Load tables
df_orders = pd.read_sql('SELECT * FROM "farmlinkApp_productorder";', engine)
df_products = pd.read_sql('SELECT * FROM "farmlinkApp_product";', engine)

# join data
df = df_orders.merge(df_products, left_on="product_id_id", right_on="id", suffixes=("_order", "_product"))

# Extract month from order date
df['month'] = pd.to_datetime(df['created_at']).dt.month

# Aggregate per crop & month
df_grouped = df.groupby(
    ['product_name', 'month']
).agg(
    total_sold=('amount', 'sum'),
    avg_price=('amount', 'mean'),
    order_count=('id_order', 'count')
).reset_index()

print(df_grouped)

# prepare features - encode crop names for ML
df_grouped['crop_id'] = df_grouped['product_name'].astype('category').cat.codes

X = df_grouped[['crop_id', 'month', 'avg_price', 'order_count']]
y = df_grouped['total_sold']

# Train Model
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train,y_train)

# Evaluate model
y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("Model Trained")
print(f"Mean Absolute Error: {mae:.2f}")
print(f"R Score: {r2:.2f}")

results = X_test.copy()
results['actual_total_sold'] = y_test
results['predicted_total_sold'] = y_pred
print("Sample Predictions:")
print(results.head())

# save model
# joblib.dump(model, 'crop_demand_model.pkl')


  product_name  month  total_sold  avg_price  order_count
0        Beans      8      1200.0      600.0            2
1  Blueberries      8      3600.0     1800.0            2
2      Carrots      8      1200.0      400.0            3
3        Maize      8       250.0      250.0            1
4       Onions      8      3600.0     3600.0            1
Model Trained
Mean Absolute Error: 2504.50
R Score: nan
Sample Predictions:
   crop_id  month  avg_price  order_count  actual_total_sold  \
1        1      8     1800.0            2             3600.0   

   predicted_total_sold  
1                1095.5  


C:\Users\jtrip\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_regression.py:1266: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)
